In [1]:
import pandas as pd

In [2]:
src_dir = "source"
out_dir = "processed"

In [3]:
filepath = {
    "cab": src_dir+"/current_account_balance/API_BN.CAB.XOKA.GD.ZS_DS2_en_csv_v2_85157.csv",
    "gdp_percap_growth": src_dir+"/gdp_per_capita_annual_growth/API_NY.GDP.PCAP.KD.ZG_DS2_en_csv_v2_85132.csv",
    "inflation": src_dir+"/inflation/API_FP.CPI.TOTL.ZG_DS2_en_csv_v2_132014.csv",
    "unemployment": src_dir+"/unemployment_rate/API_SL.UEM.TOTL.ZS_DS2_en_csv_v2_131985.csv"
}

In [4]:
def load_data(filepath, value_name):
    # The "Unnamed: 69" comes from the extra comma at the end of the lines
    df = pd.read_csv(filepath, skiprows=3)\
            .drop(columns=["Country Code","Indicator Name","Indicator Code","Unnamed: 69"])\
            .rename(columns={"Country Name": "country"})
    
    # Unpivot DataFrame (columns=[country,year1,...,yearN]) => columns=[country,year,value_name])
    df = pd.melt(df,id_vars="country",var_name="year",value_name=value_name).dropna()
    df["year"] = df["year"].astype("int")
    
    return df

# Current Account Balance

In [5]:
cab = load_data(filepath["cab"], value_name="cab")
cab.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7442 entries, 35 to 17023
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   country  7442 non-null   object 
 1   year     7442 non-null   int32  
 2   cab      7442 non-null   float64
dtypes: float64(1), int32(1), object(1)
memory usage: 203.5+ KB


In [6]:
cab_countries = cab["country"].sort_values().unique()
cab_countries

array(['Afghanistan', 'Albania', 'Algeria', 'Andorra', 'Angola',
       'Antigua and Barbuda', 'Argentina', 'Armenia', 'Aruba',
       'Australia', 'Austria', 'Azerbaijan', 'Bahamas, The', 'Bahrain',
       'Bangladesh', 'Barbados', 'Belarus', 'Belgium', 'Belize', 'Benin',
       'Bermuda', 'Bhutan', 'Bolivia', 'Bosnia and Herzegovina',
       'Botswana', 'Brazil', 'Brunei Darussalam', 'Bulgaria',
       'Burkina Faso', 'Burundi', 'Cabo Verde', 'Cambodia', 'Cameroon',
       'Canada', 'Cayman Islands', 'Central African Republic', 'Chad',
       'Chile', 'China', 'Colombia', 'Comoros', 'Congo, Dem. Rep.',
       'Congo, Rep.', 'Costa Rica', "Cote d'Ivoire", 'Croatia', 'Curacao',
       'Cyprus', 'Czechia', 'Denmark', 'Djibouti', 'Dominica',
       'Dominican Republic', 'Ecuador', 'Egypt, Arab Rep.', 'El Salvador',
       'Equatorial Guinea', 'Eritrea', 'Estonia', 'Eswatini', 'Ethiopia',
       'Faroe Islands', 'Fiji', 'Finland', 'France', 'French Polynesia',
       'Gabon', 'Gambia, The

Some of the "countries" aren't actually countries (e.g., the Faroe Islands), but that's okay.

# GDP per Capita Annual Growth

In [7]:
gdp_percap_growth = load_data(filepath["gdp_percap_growth"], value_name="gdp_percap_growth")
gdp_percap_growth.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13883 entries, 267 to 17023
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   country            13883 non-null  object 
 1   year               13883 non-null  int32  
 2   gdp_percap_growth  13883 non-null  float64
dtypes: float64(1), int32(1), object(1)
memory usage: 379.6+ KB


In [8]:
gdp_countries = gdp_percap_growth["country"].sort_values().unique()
gdp_countries

array(['Afghanistan', 'Africa Eastern and Southern',
       'Africa Western and Central', 'Albania', 'Algeria',
       'American Samoa', 'Andorra', 'Angola', 'Antigua and Barbuda',
       'Arab World', 'Argentina', 'Armenia', 'Aruba', 'Australia',
       'Austria', 'Azerbaijan', 'Bahamas, The', 'Bahrain', 'Bangladesh',
       'Barbados', 'Belarus', 'Belgium', 'Belize', 'Benin', 'Bermuda',
       'Bhutan', 'Bolivia', 'Bosnia and Herzegovina', 'Botswana',
       'Brazil', 'Brunei Darussalam', 'Bulgaria', 'Burkina Faso',
       'Burundi', 'Cabo Verde', 'Cambodia', 'Cameroon', 'Canada',
       'Caribbean small states', 'Cayman Islands',
       'Central African Republic', 'Central Europe and the Baltics',
       'Chad', 'Channel Islands', 'Chile', 'China', 'Colombia', 'Comoros',
       'Congo, Dem. Rep.', 'Congo, Rep.', 'Costa Rica', "Cote d'Ivoire",
       'Croatia', 'Cuba', 'Curacao', 'Cyprus', 'Czechia', 'Denmark',
       'Djibouti', 'Dominica', 'Dominican Republic',
       'Early-demogr

This time, there are non-countries that we really should remove (e.g., _categories_ like "Low & middle income").

In [9]:
# candidates for removal ("countries" that not in both)
set(gdp_countries)-set(cab_countries)

{'Africa Eastern and Southern',
 'Africa Western and Central',
 'American Samoa',
 'Arab World',
 'Caribbean small states',
 'Central Europe and the Baltics',
 'Channel Islands',
 'Cuba',
 'Early-demographic dividend',
 'East Asia & Pacific',
 'East Asia & Pacific (IDA & IBRD countries)',
 'East Asia & Pacific (excluding high income)',
 'Euro area',
 'Europe & Central Asia',
 'Europe & Central Asia (IDA & IBRD countries)',
 'Europe & Central Asia (excluding high income)',
 'European Union',
 'Fragile and conflict affected situations',
 'Greenland',
 'Guam',
 'Heavily indebted poor countries (HIPC)',
 'High income',
 'IBRD only',
 'IDA & IBRD total',
 'IDA blend',
 'IDA only',
 'IDA total',
 'Isle of Man',
 'Late-demographic dividend',
 'Latin America & Caribbean',
 'Latin America & Caribbean (excluding high income)',
 'Latin America & the Caribbean (IDA & IBRD countries)',
 'Least developed countries: UN classification',
 'Liechtenstein',
 'Low & middle income',
 'Low income',
 'Lower 

In [10]:
to_remove = {
    'Africa Eastern and Southern',
    'Africa Western and Central',
    # 'American Samoa',  # territory
    'Arab World',
    'Caribbean small states',
    'Central Europe and the Baltics',
    # 'Channel Islands',  # British Crown dependency
    # 'Cuba',  # country
    'Early-demographic dividend',
    'East Asia & Pacific',
    'East Asia & Pacific (IDA & IBRD countries)',
    'East Asia & Pacific (excluding high income)',
    'Euro area',
    'Europe & Central Asia',
    'Europe & Central Asia (IDA & IBRD countries)',
    'Europe & Central Asia (excluding high income)',
    'European Union',
    'Fragile and conflict affected situations',
    # 'Greenland',  # autonomous territory of Denmark
    # 'Guam',  # U.S. territory
    'Heavily indebted poor countries (HIPC)',
    'High income',
    'IBRD only',
    'IDA & IBRD total',
    'IDA blend',
    'IDA only',
    'IDA total',
    # 'Isle of Man',  # British Crown dependency
    'Late-demographic dividend',
    'Latin America & Caribbean',
    'Latin America & Caribbean (excluding high income)',
    'Latin America & the Caribbean (IDA & IBRD countries)',
    'Least developed countries: UN classification',
    # 'Liechtenstein',  # country
    'Low & middle income',
    'Low income',
    'Lower middle income',
    'Middle East & North Africa',
    'Middle East & North Africa (IDA & IBRD countries)',
    'Middle East & North Africa (excluding high income)',
    'Middle income',
    # 'Monaco',  # country
    'North America',
    # 'Northern Mariana Islands',  # U.S. territory
    'OECD members',
    'Other small states',
    'Pacific island small states',
    'Post-demographic dividend',
    'Pre-demographic dividend',
    # 'Puerto Rico',  # U.S. territory
    'Small states',
    # 'Somalia',  # country
    'South Asia',
    'South Asia (IDA & IBRD)',
    # 'St. Martin (French part)',  # French territory
    'Sub-Saharan Africa',
    'Sub-Saharan Africa (IDA & IBRD countries)',
    'Sub-Saharan Africa (excluding high income)',
    # 'Turkmenistan',  # country
    # 'United Arab Emirates',  # country
    'Upper middle income',
    # 'Virgin Islands (U.S.)',  # U.S. territory
    'World'
}

In [11]:
gdp_percap_growth = gdp_percap_growth[~gdp_percap_growth.country.isin(to_remove)]
gdp_countries = gdp_percap_growth["country"].sort_values().unique()

# Inflation

In [12]:
inflation = load_data(filepath["inflation"], value_name="inflation")
inflation.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11182 entries, 13 to 17287
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    11182 non-null  object 
 1   year       11182 non-null  int32  
 2   inflation  11182 non-null  float64
dtypes: float64(1), int32(1), object(1)
memory usage: 305.8+ KB


In [13]:
inflation_countries = inflation["country"].sort_values().unique()
inflation_countries

array(['Afghanistan', 'Africa Eastern and Southern',
       'Africa Western and Central', 'Albania', 'Algeria', 'Angola',
       'Antigua and Barbuda', 'Arab World', 'Armenia', 'Aruba',
       'Australia', 'Austria', 'Azerbaijan', 'Bahamas, The', 'Bahrain',
       'Bangladesh', 'Barbados', 'Belarus', 'Belgium', 'Belize', 'Benin',
       'Bhutan', 'Bolivia', 'Bosnia and Herzegovina', 'Botswana',
       'Brazil', 'Brunei Darussalam', 'Bulgaria', 'Burkina Faso',
       'Burundi', 'Cabo Verde', 'Cambodia', 'Cameroon', 'Canada',
       'Caribbean small states', 'Cayman Islands',
       'Central African Republic', 'Central Europe and the Baltics',
       'Chad', 'Chile', 'China', 'Colombia', 'Comoros',
       'Congo, Dem. Rep.', 'Congo, Rep.', 'Costa Rica', "Cote d'Ivoire",
       'Croatia', 'Curacao', 'Cyprus', 'Czechia', 'Denmark', 'Djibouti',
       'Dominica', 'Dominican Republic', 'Early-demographic dividend',
       'East Asia & Pacific',
       'East Asia & Pacific (IDA & IBRD countri

In [14]:
set(inflation_countries)-set(gdp_countries)

{'Africa Eastern and Southern',
 'Africa Western and Central',
 'Arab World',
 'Caribbean small states',
 'Central Europe and the Baltics',
 'Early-demographic dividend',
 'East Asia & Pacific',
 'East Asia & Pacific (IDA & IBRD countries)',
 'East Asia & Pacific (excluding high income)',
 'Euro area',
 'Europe & Central Asia',
 'Europe & Central Asia (IDA & IBRD countries)',
 'Europe & Central Asia (excluding high income)',
 'European Union',
 'Fragile and conflict affected situations',
 'Heavily indebted poor countries (HIPC)',
 'High income',
 'IBRD only',
 'IDA & IBRD total',
 'IDA blend',
 'IDA only',
 'IDA total',
 'Late-demographic dividend',
 'Latin America & Caribbean',
 'Latin America & Caribbean (excluding high income)',
 'Latin America & the Caribbean (IDA & IBRD countries)',
 'Least developed countries: UN classification',
 'Low & middle income',
 'Low income',
 'Lower middle income',
 'Middle East & North Africa',
 'Middle East & North Africa (IDA & IBRD countries)',
 'Mi

In [15]:
to_remove = {
    'Africa Eastern and Southern',
    'Africa Western and Central',
    'Arab World',
    'Caribbean small states',
    'Central Europe and the Baltics',
    'Early-demographic dividend',
    'East Asia & Pacific',
    'East Asia & Pacific (IDA & IBRD countries)',
    'East Asia & Pacific (excluding high income)',
    'Euro area',
    'Europe & Central Asia',
    'Europe & Central Asia (IDA & IBRD countries)',
    'Europe & Central Asia (excluding high income)',
    'European Union',
    'Fragile and conflict affected situations',
    'Heavily indebted poor countries (HIPC)',
    'High income',
    'IBRD only',
    'IDA & IBRD total',
    'IDA blend',
    'IDA only',
    'IDA total',
    'Late-demographic dividend',
    'Latin America & Caribbean',
    'Latin America & Caribbean (excluding high income)',
    'Latin America & the Caribbean (IDA & IBRD countries)',
    'Least developed countries: UN classification',
    'Low & middle income',
    'Low income',
    'Lower middle income',
    'Middle East & North Africa',
    'Middle East & North Africa (IDA & IBRD countries)',
    'Middle East & North Africa (excluding high income)',
    'Middle income',
    'North America',
    'OECD members',
    'Other small states',
    'Pacific island small states',
    'Post-demographic dividend',
    'Pre-demographic dividend',
    'Small states',
    'South Asia',
    'South Asia (IDA & IBRD)',
    'Sub-Saharan Africa',
    'Sub-Saharan Africa (IDA & IBRD countries)',
    'Sub-Saharan Africa (excluding high income)',
    'Upper middle income',
    'World'
}

In [16]:
inflation = inflation[~inflation.country.isin(to_remove)]
inflation_countries = inflation["country"].sort_values().unique()

# Unemployment Rate

In [17]:
unemployment = load_data(filepath["unemployment"], value_name="unemployment")
unemployment.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7981 entries, 8247 to 17289
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   country       7981 non-null   object 
 1   year          7981 non-null   int32  
 2   unemployment  7981 non-null   float64
dtypes: float64(1), int32(1), object(1)
memory usage: 218.2+ KB


In [18]:
unemployment_countries = unemployment["country"].sort_values().unique()
unemployment_countries

array(['Afghanistan', 'Africa Eastern and Southern',
       'Africa Western and Central', 'Albania', 'Algeria', 'Angola',
       'Arab World', 'Argentina', 'Armenia', 'Australia', 'Austria',
       'Azerbaijan', 'Bahamas, The', 'Bahrain', 'Bangladesh', 'Barbados',
       'Belarus', 'Belgium', 'Belize', 'Benin', 'Bhutan', 'Bolivia',
       'Bosnia and Herzegovina', 'Botswana', 'Brazil',
       'Brunei Darussalam', 'Bulgaria', 'Burkina Faso', 'Burundi',
       'Cabo Verde', 'Cambodia', 'Cameroon', 'Canada',
       'Caribbean small states', 'Central African Republic',
       'Central Europe and the Baltics', 'Chad', 'Channel Islands',
       'Chile', 'China', 'Colombia', 'Comoros', 'Congo, Dem. Rep.',
       'Congo, Rep.', 'Costa Rica', "Cote d'Ivoire", 'Croatia', 'Cuba',
       'Cyprus', 'Czechia', 'Denmark', 'Djibouti', 'Dominican Republic',
       'Early-demographic dividend', 'East Asia & Pacific',
       'East Asia & Pacific (IDA & IBRD countries)',
       'East Asia & Pacific (exclu

In [19]:
set(unemployment_countries)-set(gdp_countries)

{'Africa Eastern and Southern',
 'Africa Western and Central',
 'Arab World',
 'Caribbean small states',
 'Central Europe and the Baltics',
 'Early-demographic dividend',
 'East Asia & Pacific',
 'East Asia & Pacific (IDA & IBRD countries)',
 'East Asia & Pacific (excluding high income)',
 'Euro area',
 'Europe & Central Asia',
 'Europe & Central Asia (IDA & IBRD countries)',
 'Europe & Central Asia (excluding high income)',
 'European Union',
 'Fragile and conflict affected situations',
 'Heavily indebted poor countries (HIPC)',
 'High income',
 'IBRD only',
 'IDA & IBRD total',
 'IDA blend',
 'IDA only',
 'IDA total',
 "Korea, Dem. People's Rep.",
 'Late-demographic dividend',
 'Latin America & Caribbean',
 'Latin America & Caribbean (excluding high income)',
 'Latin America & the Caribbean (IDA & IBRD countries)',
 'Least developed countries: UN classification',
 'Low & middle income',
 'Low income',
 'Lower middle income',
 'Middle East & North Africa',
 'Middle East & North Africa

In [20]:
to_remove = {
    'Africa Eastern and Southern',
    'Africa Western and Central',
    'Arab World',
    'Caribbean small states',
    'Central Europe and the Baltics',
    'Early-demographic dividend',
    'East Asia & Pacific',
    'East Asia & Pacific (IDA & IBRD countries)',
    'East Asia & Pacific (excluding high income)',
    'Euro area',
    'Europe & Central Asia',
    'Europe & Central Asia (IDA & IBRD countries)',
    'Europe & Central Asia (excluding high income)',
    'European Union',
    'Fragile and conflict affected situations',
    'Heavily indebted poor countries (HIPC)',
    'High income',
    'IBRD only',
    'IDA & IBRD total',
    'IDA blend',
    'IDA only',
    'IDA total',
    # "Korea, Dem. People's Rep.",  North Korea
    'Late-demographic dividend',
    'Latin America & Caribbean',
    'Latin America & Caribbean (excluding high income)',
    'Latin America & the Caribbean (IDA & IBRD countries)',
    'Least developed countries: UN classification',
    'Low & middle income',
    'Low income',
    'Lower middle income',
    'Middle East & North Africa',
    'Middle East & North Africa (IDA & IBRD countries)',
    'Middle East & North Africa (excluding high income)',
    'Middle income',
    'North America',
    'OECD members',
    'Other small states',
    'Pacific island small states',
    'Post-demographic dividend',
    'Pre-demographic dividend',
    'Small states',
    'South Asia',
    'South Asia (IDA & IBRD)',
    'Sub-Saharan Africa',
    'Sub-Saharan Africa (IDA & IBRD countries)',
    'Sub-Saharan Africa (excluding high income)',
    'Upper middle income',
    'World'
}

In [21]:
unemployment = unemployment[~unemployment.country.isin(to_remove)]

# Merge

In [22]:
df = cab.merge(gdp_percap_growth, how="outer", on=["country","year"])\
        .merge(inflation, how="outer", on=["country","year"])\
        .merge(unemployment, how="outer", on=["country","year"])

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11651 entries, 0 to 11650
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   country            11651 non-null  object 
 1   year               11651 non-null  int32  
 2   cab                7442 non-null   float64
 3   gdp_percap_growth  10999 non-null  float64
 4   inflation          8904 non-null   float64
 5   unemployment       6349 non-null   float64
dtypes: float64(4), int32(1), object(1)
memory usage: 500.8+ KB


In [24]:
df.to_csv(out_dir+"/data.csv", index=False)